# 🧩 LEGO Vision AI — Pipeline v6.0 (Lightning AI)

**Arquitectura Híbrida**: Imágenes generadas localmente (Mac M4) → Entrenamiento en Cloud (GPU)

| Celda | Fase | Descripción |
|-------|------|-------------|
| C0 | Setup | Configuración del entorno Lightning AI |
| C1 | Dataset | Descomprimir dataset pre-renderizado |
| C2 | Train | Entrenamiento YOLO (Universal Detector) |
| C3 | Index | Generación de índice vectorial (FAISS) |
| C4 | Sync | Empaquetado y sincronización de resultados |


In [ ]:
# =================================================================
# CELDA 0: Setup Lightning AI (v6.0 — Hybrid Pipeline)
# =================================================================
import os, sys, json, shutil, logging, zipfile, urllib.request

# --- Find PROJECT_ROOT ---
PROJECT_ROOT = ''
for parent in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(parent, 'src')):
        PROJECT_ROOT = parent
        break
if not PROJECT_ROOT:
    PROJECT_ROOT = os.getcwd()

WORKSPACE_DIR = os.path.join(os.getcwd(), 'lightning_workspace')
DATASET_DIR = os.path.join(WORKSPACE_DIR, 'datasets')
os.makedirs(WORKSPACE_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

sys.path.insert(0, PROJECT_ROOT)

# --- Install dependencies ---
os.system('pip install ultralytics pandas requests scikit-learn psutil pyyaml faiss-cpu google-api-python-client google-auth-oauthlib -q')

# --- Logging ---
LOG_FILE_PATH = os.path.join(WORKSPACE_DIR, 'pipeline_exec.log')
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[logging.FileHandler(LOG_FILE_PATH), logging.StreamHandler()])
logger = logging.getLogger('LegoVision')

# --- Load Launch Config ---
LAUNCH_CONFIG = {}
config_path = os.path.join(PROJECT_ROOT, 'config_train.json')
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        LAUNCH_CONFIG = json.load(f)
    logger.info(f'📋 Config loaded: {LAUNCH_CONFIG.get("session_reference", "unknown")}')

RENDER_ENGINE = 'CYCLES'  # Informational only (rendering was done locally)
SET_ID = LAUNCH_CONFIG.get('session_reference', 'custom').split('_')[0] if LAUNCH_CONFIG else 'custom'

# --- Pipeline Timer (optional) ---
try:
    from src.utils.pipeline_timer import PipelineTimer
    timer = PipelineTimer()
    timer.detect_hardware()
except ImportError:
    class DummyTimer:
        def detect_hardware(self): pass
        def log_config(self, c): pass
        def step(self, name):
            class Ctx:
                def __enter__(s): return s
                def __exit__(s, *a): pass
            return Ctx()
        def save_report(self, p): pass
    timer = DummyTimer()

# --- Drive Credentials (copy if present) ---
for cred_file in ['token_1973.pickle', 'credentials.json']:
    src = os.path.join(PROJECT_ROOT, cred_file)
    dst = os.path.join(WORKSPACE_DIR, cred_file)
    if os.path.exists(src) and not os.path.exists(dst):
        shutil.copy(src, dst)

logger.info(f'Pipeline v6.0 (Hybrid) | Root: {PROJECT_ROOT} | Workspace: {WORKSPACE_DIR}')


In [ ]:
# =================================================================
# CELDA 1: 📦 Cargar Dataset Pre-Renderizado (Local M4)
# =================================================================
import os, zipfile, json, glob

logger.info('>>> FASE 1/4: Carga del Dataset Pre-Renderizado')

# Strategy: Find the dataset ZIP in the project root or workspace
zip_candidates = []
for search_dir in [PROJECT_ROOT, WORKSPACE_DIR, os.getcwd()]:
    zip_candidates.extend(glob.glob(os.path.join(search_dir, 'lightning_dataset_*.zip')))

if not zip_candidates:
    raise FileNotFoundError('❌ No se encontró ningún lightning_dataset_*.zip. Genera el dataset localmente primero.')

# Use the most recent one
dataset_zip = max(zip_candidates, key=os.path.getmtime)
logger.info(f'📦 Dataset encontrado: {os.path.basename(dataset_zip)}')

# Extract to workspace
extract_dir = WORKSPACE_DIR
with zipfile.ZipFile(dataset_zip, 'r') as z:
    z.extractall(extract_dir)
    logger.info(f'✅ Extraído en {extract_dir}')

# Find render_local directory (it should be inside the zip)
RENDER_LOCAL = os.path.join(extract_dir, 'render_local')
if not os.path.exists(RENDER_LOCAL):
    # Maybe extracted directly
    for d in os.listdir(extract_dir):
        if os.path.isdir(os.path.join(extract_dir, d)) and d != 'src':
            RENDER_LOCAL = os.path.join(extract_dir, d)
            break

# Read manifest
manifest_path = os.path.join(RENDER_LOCAL, 'dataset_manifest.json')
if os.path.exists(manifest_path):
    with open(manifest_path, 'r') as f:
        manifest = json.load(f)
    print(f'📋 Manifest: {manifest["total_pieces"]} piezas | {manifest["total_images"]} imágenes')
    for p in manifest.get('pieces', []):
        print(f'   • {p["piece_id"]}: {p["images"]} imgs, {p["labels"]} labels')
else:
    logger.warning('⚠️ No se encontró dataset_manifest.json')

# Build unified dataset by merging all piece subfolders
UNIFIED_DATASET = os.path.join(DATASET_DIR, SET_ID)
unified_images = os.path.join(UNIFIED_DATASET, 'images')
unified_labels = os.path.join(UNIFIED_DATASET, 'labels')
os.makedirs(unified_images, exist_ok=True)
os.makedirs(unified_labels, exist_ok=True)

import shutil
total_merged = 0
for piece_dir in sorted(os.listdir(RENDER_LOCAL)):
    piece_path = os.path.join(RENDER_LOCAL, piece_dir)
    if not os.path.isdir(piece_path): continue
    img_dir = os.path.join(piece_path, 'images')
    lbl_dir = os.path.join(piece_path, 'labels')
    if not os.path.exists(img_dir): continue
    
    for img in os.listdir(img_dir):
        # Prefix with piece_id to avoid collisions
        new_name = f'{piece_dir}_{img}'
        shutil.copy2(os.path.join(img_dir, img), os.path.join(unified_images, new_name))
        # Copy matching label
        lbl_name = img.replace('.jpg', '.txt').replace('.png', '.txt')
        lbl_path = os.path.join(lbl_dir, lbl_name)
        if os.path.exists(lbl_path):
            shutil.copy2(lbl_path, os.path.join(unified_labels, f'{piece_dir}_{lbl_name}'))
        total_merged += 1

# Generate unified data.yaml
data_yaml = os.path.join(UNIFIED_DATASET, 'data.yaml')
with open(data_yaml, 'w') as f:
    f.write(f'path: {UNIFIED_DATASET}')
    f.write('train: images')
    f.write('val: images')
    f.write('nc: 1')
    f.write("names: ['lego']")

logger.info(f'✅ Dataset unificado: {total_merged} imágenes en {UNIFIED_DATASET}')
# Make PARTS_TO_TRAIN available for downstream cells
PARTS_TO_TRAIN = [{'ldraw_id': p['piece_id'], 'name': p['piece_id']} for p in manifest.get('pieces', [])] if 'manifest' in dir() else [{'ldraw_id': SET_ID, 'name': SET_ID}]


In [ ]:
# =================================================================
# CELDA 2: 🏋️ YOLO Training (Universal Detector)
# =================================================================
from ultralytics import YOLO
import torch, time as _time, glob, shutil, datetime

logger.info('>>> FASE 2/4: Entrenamiento YOLO')

dataset_path = os.path.join(DATASET_DIR, SET_ID)
data_yaml = os.path.join(dataset_path, 'data.yaml')

if not os.path.exists(data_yaml):
    raise FileNotFoundError(f'❌ No se encontró data.yaml en {dataset_path}')

results_dir = os.path.join(WORKSPACE_DIR, 'results')

# Progressive Training: use latest weights if available
base_model_path = 'yolo11s-seg.pt'
models_dir = os.path.join(PROJECT_ROOT, 'models', 'yolo_model')
os.makedirs(models_dir, exist_ok=True)

trained_weights = sorted(glob.glob(os.path.join(models_dir, 'universal_detector_*.pt')))
if trained_weights:
    base_model_path = trained_weights[-1]
    print(f'🧠 Incremental training from: {os.path.basename(base_model_path)}')
else:
    print('🌱 Training from base (yolo11s-seg.pt)')

model = YOLO(base_model_path, task='segment')

num_gpus = torch.cuda.device_count()
train_device = list(range(num_gpus)) if num_gpus > 1 else (0 if num_gpus == 1 else 'cpu')

if torch.cuda.is_available():
    os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
    torch.cuda.empty_cache()

# Callback for epoch logging
_t_epoch = [_time.time()]
def _on_epoch_end(trainer):
    elapsed = _time.time() - _t_epoch[0]
    m = trainer.metrics or {}
    vram = torch.cuda.memory_reserved(0)/1e9 if torch.cuda.is_available() else 0
    print(f'[EPOCH {trainer.epoch+1}/{trainer.epochs}] {elapsed:.1f}s | mAP50: {m.get("metrics/mAP50(B)",0):.4f} | VRAM: {vram:.1f}GB', flush=True)
model.add_callback('on_fit_epoch_end', _on_epoch_end)

TRAIN_EPOCHS = 150
print(f'🚀 Training on: {train_device} | Epochs: {TRAIN_EPOCHS}')

with timer.step('YOLO Training'):
    model.train(
        task='segment',
        data=data_yaml,
        epochs=TRAIN_EPOCHS,
        patience=10,
        imgsz=640,
        project=results_dir,
        name=f'yolo11_{SET_ID}',
        batch=-1,
        device=train_device,
        workers=4,
        cache=True,
        amp=True,
        optimizer='auto',
        verbose=False,
        mosaic=0.7,
        mixup=0.1,
        scale=0.05,
        degrees=180.0,
        translate=0.2,
        hsv_s=0.7,
        hsv_v=0.4,
        bgr=0.2,
        blur=0.1,
        erasing=0.1
    )

# Auto-version the model
dirs = glob.glob(os.path.join(results_dir, f'yolo11_{SET_ID}*'))
latest_dir = max(dirs, key=os.path.getmtime) if dirs else ''
best_pt = os.path.join(latest_dir, 'weights', 'best.pt') if latest_dir else ''
if os.path.exists(best_pt):
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
    dest = os.path.join(models_dir, f'universal_detector_{ts}.pt')
    shutil.copy2(best_pt, dest)
    print(f'✅ Model saved: {dest}')
else:
    print('⚠️ No best.pt found')
